# 02 — EDA: genomic (synthetic)

Compare **baseline** synthetic genotype weights with the optional
**West Africa rs334 MAF** tilt (`west_africa_rs334_maf`). The latter is a
*benchmark prior only* — not a substitute for real genotypes.


In [ ]:
import sys
from pathlib import Path

_repo = Path.cwd()
for _ in range(4):
    if (_repo / "src" / "mmvlm4scd").is_dir():
        sys.path.insert(0, str(_repo / "src"))
        break
    _repo = _repo.parent

import matplotlib.pyplot as plt
import numpy as np

from mmvlm4scd.data import generate_synthetic_cohort
from mmvlm4scd.data.synthetic import SCDSyntheticConfig


In [ ]:
def genotype_props(cfg):
    c = generate_synthetic_cohort(cfg)
    g = c["clinical"]["genotype"].values
    labs, cnt = np.unique(g, return_counts=True)
    return dict(zip(labs, cnt / cnt.sum()))

base = genotype_props(SCDSyntheticConfig(n_patients=8000, seed=0))
wa = genotype_props(SCDSyntheticConfig(n_patients=8000, seed=0, west_africa_rs334_maf=0.13))

labels = sorted(set(base) | set(wa))
x = np.arange(len(labels))
w = 0.35
plt.bar(x - w / 2, [base[k] for k in labels], width=w, label="baseline")
plt.bar(x + w / 2, [wa[k] for k in labels], width=w, label="west_africa_rs334_maf=0.13")
plt.xticks(x, labels, rotation=30, ha="right")
plt.ylabel("fraction")
plt.title("Synthetic genotype mix: baseline vs West Africa MAF tilt")
plt.legend()
plt.tight_layout()
plt.show()


## Variant-indicator block (first 16 dims)

Column 0 is aligned with higher genotype severity in the simulator.


In [ ]:
cohort = generate_synthetic_cohort(SCDSyntheticConfig(n_patients=2000, seed=1))
g = cohort["genomic"]
print("genomic shape (N, 32):", g.shape)
print("mean allele-indicator dims 0–15:", g[:, :16].mean(axis=0)[:8])
